# GRN calibration experiments: setup

This lightweight notebook prepares the shared helper module and checks the runtime. Run it first in Google Colab. Upload/extract the complete project ZIP into `/content/GRN_Calibration_Experiments` or place this notebook and `grn_experiment_utils.py` in the same folder.

The experiment uses a small Transformer encoder over paired TF/target expression profiles. It compares predictive-only BCE (`lambda=1`) against end-to-end calibration (`lambda<1`) using the same architecture. This is an experimental baseline, not a claim that attention weights establish causal regulation.

In [1]:
# Colab setup. CPU is sufficient for the default small runs; GPU can speed them up.
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn nbformat

In [2]:
import os, sys, torch
import numpy as np
import pandas as pd
from google.colab import drive

In [3]:
drive.mount('/content/drive/', force_remount=True)
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/"
if not os.path.isdir(PROJECT_DIR):
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
print("Project directory:", PROJECT_DIR)
print("PyTorch:", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive/
Project directory: /content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/
PyTorch: 2.11.0+cpu | device: cpu


In [4]:
# Verify the shared module is visible.
from grn_experiment_utils import PairTransformer, soft_calibration_penalty, evaluate

model = PairTransformer()
x = torch.randn(4, 32, 2)
print("Transformer output shape:", tuple(model(x).shape))
print("Calibration penalty smoke test:", float(soft_calibration_penalty(model(x), torch.tensor([0.,1.,0.,1.]))))

/content/drive/MyDrive/Colab Notebooks/Thesis/Working Code/GRN_Calibration_Experiments/grn_experiment_utils.py:82: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder=nn.TransformerEncoder(enc_layer, num_layers=layers)


Transformer output shape: (4,)
Calibration penalty smoke test: 0.0625750944018364


/tmp/ipykernel_976/507808855.py:7: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("Calibration penalty smoke test:", float(soft_calibration_penalty(model(x), torch.tensor([0.,1.,0.,1.]))))


## Practical limits

Default settings cap each pair profile at 64 conditions and the DREAM5 candidate set at 12,000 directed pairs. Start with 20–30 epochs and one SERGIO network. Increase the cap only after a successful run. The split is by directed gene pair, not by individual cells/conditions; results therefore assess held-out edge prediction within the same set of genes, not generalisation to unseen genes.